In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,RandomizedSearchCV,GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier,AdaBoostClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix,mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from catboost import CatBoostRegressor, CatBoostClassifier
from xgboost import XGBRegressor, XGBClassifier

In [2]:
#import the csv
df = pd.read_csv('data/salary.csv')

In [3]:
X = df.drop('Salary', axis=1)

In [4]:
y=df['Salary']

In [5]:
# Convert column to category type, then grab the integer codes
all_mappings = {}
for col in X.columns:
    if X[col].dtype != 'int64' and X[col].dtype != 'float64':
        X[col] = X[col].astype('category')
        all_mappings[col] = dict(enumerate(X[col].cat.categories))
        X[col] = X[col].astype('category').cat.codes

X.head()
print(all_mappings)

{'Gender': {0: 'Female', 1: 'Male'}, 'Job Title': {0: 'Account Executive', 1: 'Account Manager', 2: 'Accountant', 3: 'Administrative Assistant', 4: 'Advertising Coordinator', 5: 'Back end Developer', 6: 'Business Analyst', 7: 'Business Development Associate', 8: 'Business Development Manager', 9: 'Business Intelligence Analyst', 10: 'Business Operations Analyst', 11: 'CEO', 12: 'Chief Data Officer', 13: 'Chief Technology Officer', 14: 'Consultant', 15: 'Content Marketing Manager', 16: 'Copywriter', 17: 'Creative Director', 18: 'Customer Service Manager', 19: 'Customer Service Rep', 20: 'Customer Service Representative', 21: 'Customer Success Manager', 22: 'Customer Success Rep', 23: 'Customer Support Specialist', 24: 'Data Analyst', 25: 'Data Engineer', 26: 'Data Entry Clerk', 27: 'Data Scientist', 28: 'Delivery Driver', 29: 'Designer', 30: 'Developer', 31: 'Digital Content Producer', 32: 'Digital Marketing Manager', 33: 'Digital Marketing Specialist', 34: 'Director', 35: 'Director of 

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5347, 8), (1337, 8), (5347,), (1337,))

### Evaluate Model

In [7]:
def evaluate_model(true,predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, predicted)
    return mae, mse, rmse, r2

In [21]:
models = {"Linear Regression": LinearRegression(),
          "Ridge Regression": Ridge(),
          "Lasso Regression": Lasso(),
          "Support Vector Regression": SVR(),
          "Random Forest Regression": RandomForestRegressor(),
          "K-Nearest Neighbors Regression": KNeighborsRegressor(),
          "Decision Tree Regression": DecisionTreeRegressor(),
          "CatBoost Regressor": CatBoostRegressor(),
          "XGBoost Regressor": XGBRegressor()
          }

models_list=[]
r2_list=[]

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train) #train model

    #make predictions
    y_train_pred=model.predict(X_train)
    y_test_pred=model.predict(X_test)

    #evaluate model
    model_train_mae, model_train_mse, model_train_rmse, model_train_r2 = evaluate_model(y_train,y_train_pred)
    model_test_mae, model_test_mse, model_test_rmse, model_test_r2 = evaluate_model(y_test,y_test_pred)

    print(list(models.keys())[i])
    models_list.append(list(models.keys())[i])
    r2_list.append(model_test_r2)

    print("Model performance on Training set")
    print(f"MAE: {model_train_mae}, MSE: {model_train_mse}, RMSE: {model_train_rmse}, R2: {model_train_r2}")
    print("--------------------------------")
    print("Model performance on Test set")
    print(f"MAE: {model_test_mae}, MSE: {model_test_mse}, RMSE: {model_test_rmse}, R2: {model_test_r2}")

Linear Regression
Model performance on Training set
MAE: 22121.1264484044, MSE: 803811996.6742944, RMSE: 28351.578380652714, R2: 0.7119418204107051
--------------------------------
Model performance on Test set
MAE: 21800.857234816776, MSE: 796582260.0794746, RMSE: 28223.78890367972, R2: 0.7133473706424021
Ridge Regression
Model performance on Training set
MAE: 22121.59561344535, MSE: 803812040.1401073, RMSE: 28351.579147202847, R2: 0.711941804834074
--------------------------------
Model performance on Test set
MAE: 21801.345609733653, MSE: 796578912.061348, RMSE: 28223.729591628176, R2: 0.7133485754372453
Lasso Regression
Model performance on Training set
MAE: 22121.317617164375, MSE: 803812014.990818, RMSE: 28351.578703677475, R2: 0.711941813846702
--------------------------------
Model performance on Test set
MAE: 21801.055463086403, MSE: 796579760.0681837, RMSE: 28223.744614564945, R2: 0.7133482702793208
Support Vector Regression
Model performance on Training set
MAE: 45721.884314

### Results

In [22]:
pd.DataFrame(list(zip(models_list, r2_list)), columns=["Model", "R2"]).sort_values(by=["R2"], ascending=False)

,Model,R2
8,XGBoost Regressor,0.970029
4,Random Forest Regression,0.965358
7,CatBoost Regressor,0.962893
6,Decision Tree Regression,0.943545
5,K-Nearest Neighbors Regression,0.916603
1,Ridge Regression,0.713349
2,Lasso Regression,0.713348
0,Linear Regression,0.713347
3,Support Vector Regression,0.002499
